# GéoMarketing IDF — J7 : Profil détaillé de la clientèle potentielle

## 07d - Consolidation du profil de la clientèle

In [1]:
#Importation des librairies
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd
import openpyxl

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Openpyxl :", openpyxl.__version__)
print("Importations réussies ✅")

Pandas : 2.2.2
NumPy : 1.26.4
Openpyxl : 3.1.5
Importations réussies ✅


In [2]:
#Dossiers
RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_RAW = (
    RACINE
    / "data"
    / "raw"
    / "insee"
    / "rp2023"
)

DOSSIER_INTERIM = (
    RACINE
    / "data"
    / "interim"
)

DOSSIER_PROCESSED = (
    RACINE
    / "data"
    / "processed"
)

for dossier in [
    DOSSIER_RAW,
    DOSSIER_INTERIM,
    DOSSIER_PROCESSED,
]:
    dossier.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Racine :", RACINE)
print("Raw :", DOSSIER_RAW)
print("Interim :", DOSSIER_INTERIM)
print("Processed :", DOSSIER_PROCESSED)

assert RACINE.exists(), "Le dossier du projet n'existe pas."

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Raw : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\rp2023
Interim : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim
Processed : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed


In [3]:
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize(
        "NFKD",
        nom,
    )

    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )

    nom = re.sub(
        r"[^A-Z0-9]+",
        "_",
        nom,
    )

    return nom.strip("_")


def normaliser_code_commune(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.upper()
        .str.zfill(5)
    )


def pourcentage(numerateur, denominateur):
    numerateur = pd.to_numeric(
        numerateur,
        errors="coerce",
    )

    denominateur = pd.to_numeric(
        denominateur,
        errors="coerce",
    )

    return (
        numerateur
        .div(
            denominateur.where(
                denominateur.ne(0)
            )
        )
        .mul(100)
    )


def verifier_classeur_xlsx(fichier):
    if not fichier.exists():
        raise FileNotFoundError(
            f"Classeur introuvable : {fichier}"
        )

    with open(fichier, "rb") as flux:
        signature = flux.read(4)

    if signature != b"PK\x03\x04":
        raise ValueError(
            f"{fichier.name} n'est pas un véritable fichier XLSX."
        )

    with ZipFile(fichier) as archive:
        noms = set(archive.namelist())

        if "xl/workbook.xml" not in noms:
            raise ValueError(
                f"{fichier.name} ne contient pas de classeur Excel valide."
            )

    print(
        f"Classeur valide : {fichier.name} "
        f"({fichier.stat().st_size / 1_000_000:.2f} Mo)"
    )


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        sep=",",
        encoding="utf-8-sig",
    )

    print(
        "Fichier CSV créé :",
        fichier,
    )

In [4]:
noms_profils_j6 = [
    "profil_communes_idf_j6.xlsx",
    "profil_communes_idf_j6.csv",
    "profil_communes_idf.csv",
]

FICHIER_PROFIL_J6 = None

for nom in noms_profils_j6:
    candidat = DOSSIER_PROCESSED / nom

    if candidat.exists():
        FICHIER_PROFIL_J6 = candidat
        break

if FICHIER_PROFIL_J6 is None:
    raise FileNotFoundError(
        "Le profil J6 est introuvable dans data/processed."
    )

if FICHIER_PROFIL_J6.suffix.lower() == ".xlsx":
    profil_j6 = pd.read_excel(
        FICHIER_PROFIL_J6,
        sheet_name=0,
        engine="openpyxl",
    )

else:
    profil_j6 = pd.read_csv(
        FICHIER_PROFIL_J6,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
    )

profil_j6.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j6.columns
]


profil_j6.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j6.columns
]

if "CODGEO" not in profil_j6.columns:
    candidats_code = [
        "DEPCOM",
        "CODE_COMMUNE",
        "COM",
    ]

    colonne_code = next(
        (
            colonne
            for colonne in candidats_code
            if colonne in profil_j6.columns
        ),
        None,
    )

    if colonne_code is None:
        raise ValueError(
            "Aucune colonne de code communal dans le profil J6."
        )

    profil_j6 = profil_j6.rename(
        columns={
            colonne_code: "CODGEO"
        }
    )

profil_j6["CODGEO"] = normaliser_code_commune(
    profil_j6["CODGEO"]
)

assert profil_j6["CODGEO"].notna().all()
assert profil_j6["CODGEO"].is_unique
assert profil_j6["CODGEO"].str.fullmatch(
    r"\d{5}"
).all()

codes_profil = set(
    profil_j6["CODGEO"]
)

print("Profil J6 :", FICHIER_PROFIL_J6)
print("Nombre de communes :", len(profil_j6))
print("Profil J6 chargé ✅")
    

Profil J6 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j6.csv
Nombre de communes : 1266
Profil J6 chargé ✅


In [6]:
#Définir les fichiers intermédiaires

FICHIER_7A = (
    DOSSIER_INTERIM
    / "j7/j7a_population_age_sexe_idf_2023.csv"
)

FICHIER_7B = (
    DOSSIER_INTERIM
    / "j7/j7b_menages_familles_idf_2023.csv"
)

FICHIER_7C = (
    DOSSIER_INTERIM
    / "j7/j7c_scolarisation_diplomes_idf_2023.csv"
)
    

In [9]:
def charger_indicateurs(
    fichier,
    nom_bloc,
):
    table = pd.read_csv(
        fichier,
        sep=",",
        encoding="utf-8-sig",
        dtype={
            "CODGEO": "string",
        },
        low_memory=False,
    )

    table.columns = [
        normaliser_nom_colonne(colonne)
        for colonne in table.columns
    ]

    if "CODGEO" not in table.columns:
        raise ValueError(
            f"CODGEO absent du bloc {nom_bloc}."
        )

    table["CODGEO"] = (
        normaliser_code_commune(
            table["CODGEO"]
        )
    )

    assert table["CODGEO"].notna().all()
    assert table["CODGEO"].is_unique

    colonnes_libelles = [
        colonne
        for colonne in [
            "LIBGEO",
            "LIBELLE_GEOGRAPHIQUE",
        ]
        if colonne in table.columns
    ]

    table = table.drop(
        columns=colonnes_libelles,
    )

    print(
        nom_bloc,
        ":",
        table.shape,
    )

    return table

In [13]:
#Charger les trois tables

table_7a = charger_indicateurs(
    FICHIER_7A,
    "J7a",
)

table_7b = charger_indicateurs(
    FICHIER_7B,
    "J7b",
)

table_7c = charger_indicateurs(
    FICHIER_7C,
    "J7c",
)

J7a : (1266, 48)
J7b : (1266, 95)
J7c : (1266, 64)


In [14]:
#Contrôler les codes géographiques

tables_j7 = {
    "J7A_AGE_SEXE": table_7a,
    "J7B_MENAGES_FAMILLES": table_7b,
    "J7C_SCOLARISATION_DIPLOMES": table_7c,
}

lignes_controle_jointures = []

for nom_bloc, table in tables_j7.items():
    codes_table = set(
        table["CODGEO"]
    )

    codes_absents = sorted(
        codes_profil - codes_table
    )

    codes_supplementaires = sorted(
        codes_table - codes_profil
    )

    lignes_controle_jointures.append(
        {
            "BLOC": nom_bloc,
            "NB_CODES_TABLE": len(codes_table),
            "NB_CODES_ABSENTS_PROFIL": len(
                codes_absents
            ),
            "CODES_ABSENTS": ", ".join(
                codes_absents
            ),
            "NB_CODES_SUPPLEMENTAIRES": len(
                codes_supplementaires
            ),
            "CODES_SUPPLEMENTAIRES": ", ".join(
                codes_supplementaires
            ),
        }
    )

controle_jointures = pd.DataFrame(
    lignes_controle_jointures
)

display(controle_jointures)

assert (
    controle_jointures[
        "NB_CODES_ABSENTS_PROFIL"
    ]
    == 0
).all()

assert (
    controle_jointures[
        "NB_CODES_SUPPLEMENTAIRES"
    ]
    == 0
).all()

,BLOC,NB_CODES_TABLE,NB_CODES_ABSENTS_PROFIL,CODES_ABSENTS,NB_CODES_SUPPLEMENTAIRES,CODES_SUPPLEMENTAIRES
0,J7A_AGE_SEXE,1266,0,,0,
1,J7B_MENAGES_FAMILLES,1266,0,,0,
2,J7C_SCOLARISATION_DIPLOMES,1266,0,,0,


In [15]:
#Vérifier nom de colonnes avant jointure

profil_j7 = profil_j6.copy()

for nom_bloc, table in tables_j7.items():
    colonnes_communes = (
        set(profil_j7.columns)
        & set(table.columns)
    ) - {
        "CODGEO"
    }

    if colonnes_communes:
        raise ValueError(
            f"Colonnes déjà présentes avant la jointure "
            f"de {nom_bloc} : {sorted(colonnes_communes)}"
        )

print("Aucun conflit de colonnes ✅")

Aucun conflit de colonnes ✅


In [16]:
#Effectuer les jointures

nombre_communes_avant = len(
    profil_j7
)

for nom_bloc, table in tables_j7.items():
    profil_j7 = profil_j7.merge(
        table,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )

    print(
        f"{nom_bloc} joint ✅ — "
        f"{len(profil_j7)} communes"
    )

assert len(profil_j7) == nombre_communes_avant
assert profil_j7["CODGEO"].is_unique

J7A_AGE_SEXE joint ✅ — 1266 communes
J7B_MENAGES_FAMILLES joint ✅ — 1266 communes
J7C_SCOLARISATION_DIPLOMES joint ✅ — 1266 communes


In [17]:
#Contrôler les données ma,nquantes
colonnes_temoin = {
    "J7A": "POP_RP2023",
    "J7B": "NB_MENAGES",
    "J7C": "POP_NON_SCOL_15P",
}

for bloc, colonne in colonnes_temoin.items():
    profil_j7[
        f"MANQUANT_{bloc}"
    ] = profil_j7[
        colonne
    ].isna()

profil_j7[
    "NB_BLOCS_J7_MANQUANTS"
] = (
    profil_j7[
        [
            "MANQUANT_J7A",
            "MANQUANT_J7B",
            "MANQUANT_J7C",
        ]
    ]
    .sum(axis=1)
)

communes_incompletes = profil_j7[
    profil_j7[
        "NB_BLOCS_J7_MANQUANTS"
    ]
    > 0
].copy()

print(
    "Communes avec au moins un bloc manquant :",
    len(communes_incompletes),
)

display(
    communes_incompletes[
        [
            "CODGEO",
            "MANQUANT_J7A",
            "MANQUANT_J7B",
            "MANQUANT_J7C",
        ]
    ].head(20)
)

assert communes_incompletes.empty

Communes avec au moins un bloc manquant : 0


,CODGEO,MANQUANT_J7A,MANQUANT_J7B,MANQUANT_J7C


In [18]:
#Comparer avec l'ancienne population de profil
colonnes_population_j6 = [
    "POPULATION",
    "POPULATION_MUNICIPALE",
    "POPULATION_TOTALE",
    "P22_POP",
    "P21_POP",
    "POP",
]

COLONNE_POPULATION_J6 = next(
    (
        colonne
        for colonne in colonnes_population_j6
        if colonne in profil_j6.columns
    ),
    None,
)

if COLONNE_POPULATION_J6 is not None:
    population_j6_numerique = pd.to_numeric(
        profil_j7[
            COLONNE_POPULATION_J6
        ],
        errors="coerce",
    )

    profil_j7[
        "ECART_POP_RP2023_MOINS_J6"
    ] = (
        profil_j7["POP_RP2023"]
        - population_j6_numerique
    )

    profil_j7[
        "ECART_POP_RP2023_MOINS_J6_PCT"
    ] = pourcentage(
        profil_j7[
            "ECART_POP_RP2023_MOINS_J6"
        ],
        population_j6_numerique,
    )

    print(
        "Population J6 comparée avec :",
        COLONNE_POPULATION_J6,
    )

else:
    print(
        "Aucune ancienne colonne de population "
        "détectée : comparaison ignorée."
    )

Aucune ancienne colonne de population détectée : comparaison ignorée.


In [19]:
#Créer un dictionnaire de nouvelles colonnes

colonnes_initiales = set(
    profil_j6.columns
)

lignes_dictionnaire = []

for nom_bloc, table in tables_j7.items():
    for colonne in table.columns:
        if colonne == "CODGEO":
            continue

        if colonne.startswith("PART_"):
            nature = "Pourcentage"

        elif colonne.startswith("TAUX_"):
            nature = "Taux en pourcentage"

        elif colonne.startswith("ECART_"):
            nature = "Écart"

        elif colonne.startswith("POP_"):
            nature = "Effectif de population estimé"

        elif colonne.startswith("NB_"):
            nature = "Effectif estimé"

        else:
            nature = "Indicateur dérivé"

        lignes_dictionnaire.append(
            {
                "VARIABLE": colonne,
                "BLOC_SOURCE": nom_bloc,
                "NATURE": nature,
                "MILLÉSIME": "RP 2023",
            }
        )

dictionnaire_j7 = pd.DataFrame(
    lignes_dictionnaire
)

display(
    dictionnaire_j7.head(20)
)

,VARIABLE,BLOC_SOURCE,NATURE,MILLÉSIME
0,POP_0_2,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
1,POP_3_5,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
2,POP_6_10,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
3,POP_11_14,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
4,POP_15_17,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
5,POP_18_24,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
6,POP_25_39,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
7,POP_40_54,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
8,POP_55_64,J7A_AGE_SEXE,Effectif de population estimé,RP 2023
9,POP_65_79,J7A_AGE_SEXE,Effectif de population estimé,RP 2023


In [21]:
assert len(profil_j7) == len(profil_j6)
assert profil_j7["CODGEO"].is_unique
assert profil_j7["CODGEO"].notna().all()

assert profil_j7["POP_RP2023"].ge(0).all()
assert profil_j7["NB_MENAGES"].ge(0).all()
assert profil_j7["NB_FAMILLES"].ge(0).all()

colonnes_pourcentages = [
    colonne
    for colonne in profil_j7.columns
    if (
        colonne.startswith("PART_")
        or colonne.startswith("TAUX_")
    )
]



assert "75056" in profil_j7["CODGEO"].values
assert "93066" in profil_j7["CODGEO"].values
assert "93059" not in profil_j7["CODGEO"].values

print(
    "Communes finales :",
    len(profil_j7),
)

print(
    "Nombre total de colonnes :",
    len(profil_j7.columns),
)

print("Tous les contrôles J7d sont validés ✅")

Communes finales : 1266
Nombre total de colonnes : 253
Tous les contrôles J7d sont validés ✅


In [23]:
FICHIER_PROFIL_J7 = (
    DOSSIER_PROCESSED
    / "profil_communes_idf_j7.csv"
)

FICHIER_CONTROLE_J7 = (
    DOSSIER_INTERIM
    / "j7/j7d_controle_jointures.csv"
)

FICHIER_DICTIONNAIRE_J7 = (
    DOSSIER_INTERIM
    / "j7/j7d_dictionnaire_variables.csv"
)

enregistrer_csv(
    profil_j7,
    FICHIER_PROFIL_J7,
)

enregistrer_csv(
    controle_jointures,
    FICHIER_CONTROLE_J7,
)

enregistrer_csv(
    dictionnaire_j7,
    FICHIER_DICTIONNAIRE_J7,
)

print("Profil final J7 :", FICHIER_PROFIL_J7)
print("J7 terminé ✅")

Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j7.csv
Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7\j7d_controle_jointures.csv
Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7\j7d_dictionnaire_variables.csv
Profil final J7 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j7.csv
J7 terminé ✅
